In [2]:
# 00_verify_sepsis_stack.py
# I wrote this to quickly sanity-check my Sepsis pipeline without changing anything.
# It works both as a script and inside a Jupyter cell (no __file__ issues).

import json
import sys
import os
from pathlib import Path

def get_root():
    # If I'm in a notebook, __file__ doesn't exist. I fallback to CWD.
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path(os.getcwd()).resolve()

report = {
    "python": {},
    "packages": {},
    "paths": {},
    "data_checks": {},
    "model_imports": {},
    "model_loads": {},
    "rl_agent": {},
    "hints": {}
}

ROOT = get_root()
DATA_PROC = ROOT / "data" / "processed" / "sepsis"
MODELS = ROOT / "models"
SRC_MODELS = ROOT / "src" / "models"
LOGS = ROOT / "logs"

expected_files = {
    "train_X": DATA_PROC / "train_X.csv",
    "train_y": DATA_PROC / "train_y.csv",
    "val_X":   DATA_PROC / "val_X.csv",
    "val_y":   DATA_PROC / "val_y.csv",
    "test_X":  DATA_PROC / "test_X.csv",
    "test_y":  DATA_PROC / "test_y.csv",
    "scaler":  DATA_PROC / "scaler.joblib",
    "imputer": DATA_PROC / "imputer.joblib",
    "gain_generator": MODELS / "generator_sepsis.pth",
    "classifier":     MODELS / "classifier_sepsis.pth",
    "rl_agent_zip":   MODELS / "rl_agent_sepsis_masked_final.zip",
    "src_classifier_py": SRC_MODELS / "classifier.py",
    "src_gain_py":       SRC_MODELS / "gain.py",
    "src_transformer_py":SRC_MODELS / "transformer.py",
}

# Python version
report["python"]["version"] = sys.version

# Package import helper
def safe_import(name):
    try:
        mod = __import__(name, fromlist=["*"])
        ver = getattr(mod, "__version__", "unknown")
        report["packages"][name] = {"ok": True, "version": ver}
        return mod
    except Exception as e:
        report["packages"][name] = {"ok": False, "error": str(e)}
        return None

# Try imports I rely on
torch = safe_import("torch")
pd = safe_import("pandas")
np = safe_import("numpy")
joblib = safe_import("joblib")
sklearn = safe_import("sklearn")
sb3 = safe_import("stable_baselines3")

# Paths existence
for k, p in expected_files.items():
    report["paths"][k] = {"exists": p.exists(), "path": str(p)}

# Data checks
try:
    if pd is not None:
        def load_shape_nan(path):
            df = pd.read_csv(path)
            return {
                "rows": int(df.shape[0]),
                "cols": int(df.shape[1]),
                "nan_cells": int(df.isna().sum().sum())
            }
        for key in ["train_X","train_y","val_X","val_y","test_X","test_y"]:
            p = expected_files[key]
            report["data_checks"][key] = load_shape_nan(p) if p.exists() else {"error":"missing"}
except Exception as e:
    report["data_checks"]["_error"] = f"pandas_check_failed: {e}"

# Model imports
sys.path.insert(0, str(ROOT))
model_class_info = {
    "classifier": ("src.models.classifier", "PreliminaryClassifier"),
    "gain_generator": ("src.models.gain", "Generator"),
    "transformer": ("src.models.transformer", "TransformerSelector"),
}
for tag, (modname, clsname) in model_class_info.items():
    try:
        mod = __import__(modname, fromlist=[clsname])
        cls = getattr(mod, clsname, None)
        report["model_imports"][tag] = {
            "module": modname, "class": clsname,
            "ok": cls is not None,
            "detail": "found" if cls is not None else "not found (name may differ)"
        }
    except Exception as e:
        report["model_imports"][tag] = {"module": modname, "class": clsname, "ok": False, "error": str(e)}

# Try loading torch models (state_dict only)
try:
    if torch is not None:
        # Classifier
        if report["model_imports"].get("classifier",{}).get("ok") and expected_files["classifier"].exists():
            from src.models.classifier import PreliminaryClassifier as _PC
            try:
                input_dim = None
                if expected_files["train_X"].exists() and pd is not None:
                    _df = pd.read_csv(expected_files["train_X"])
                    input_dim = _df.shape[1]
                if input_dim is None:
                    input_dim = 37  # fallback to my sepsis aggregated features
                model = _PC(input_dim=input_dim)
                sd = torch.load(expected_files["classifier"], map_location="cpu")
                missing, unexpected = model.load_state_dict(sd, strict=False)
                report["model_loads"]["classifier"] = {
                    "ok": True,
                    "missing_keys": list(missing),
                    "unexpected_keys": list(unexpected)
                }
            except Exception as e:
                report["model_loads"]["classifier"] = {"ok": False, "error": str(e)}
        else:
            report["model_loads"]["classifier"] = {"ok": False, "error": "class_or_file_missing"}

        # GAIN Generator
        if report["model_imports"].get("gain_generator",{}).get("ok") and expected_files["gain_generator"].exists():
            from src.models.gain import Generator as _Gen
            try:
                input_dim = None
                if expected_files["train_X"].exists() and pd is not None:
                    _df = pd.read_csv(expected_files["train_X"])
                    input_dim = _df.shape[1]
                if input_dim is None:
                    input_dim = 37
                gen = _Gen(input_dim=input_dim, hidden_dim=128)  # I’ll adjust signature later if needed
                sd = torch.load(expected_files["gain_generator"], map_location="cpu")
                missing, unexpected = gen.load_state_dict(sd, strict=False)
                report["model_loads"]["gain_generator"] = {
                    "ok": True,
                    "missing_keys": list(missing),
                    "unexpected_keys": list(unexpected)
                }
            except Exception as e:
                report["model_loads"]["gain_generator"] = {"ok": False, "error": str(e)}
        else:
            report["model_loads"]["gain_generator"] = {"ok": False, "error": "class_or_file_missing"}

except Exception as e:
    report["model_loads"]["_error"] = f"torch_or_load_failed: {e}"

# PPO log hint
try:
    ppo_dirs = []
    if LOGS.exists():
        for p in LOGS.iterdir():
            if p.is_dir() and p.name.startswith("PPO_"):
                ppo_dirs.append(p.name)
    report["hints"]["ppo_named_runs_in_logs"] = ppo_dirs
except Exception as e:
    report["hints"]["ppo_named_runs_in_logs_error"] = str(e)

# SB3 PPO load check
try:
    if sb3 is not None:
        from stable_baselines3 import PPO
        zip_path = expected_files["rl_agent_zip"]
        if zip_path.exists():
            try:
                _ = PPO.load(str(zip_path))
                report["rl_agent"] = {"ok": True, "format": "SB3/PPO", "path": str(zip_path)}
            except Exception as e:
                report["rl_agent"] = {"ok": False, "path": str(zip_path), "error": str(e)}
        else:
            report["rl_agent"] = {"ok": False, "error": "rl_agent_zip_missing"}
    else:
        report["rl_agent"] = {"ok": False, "error": "stable_baselines3_not_installed"}
except Exception as e:
    report["rl_agent"] = {"ok": False, "error": f"sb3_check_failed: {e}"}

# Output
out_dir = ROOT / "logs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "sepsis_setup_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print("\n=== Sepsis Setup Report (summary) ===")
print(json.dumps({
    "python": report["python"],
    "packages": report["packages"],
    "paths": report["paths"],
    "data_shapes": report.get("data_checks", {}),
    "imports_ok": report.get("model_imports", {}),
    "model_loads": report.get("model_loads", {}),
    "rl_agent": report.get("rl_agent", {}),
    "hints": report.get("hints", {})
}, indent=2))
print(f"\nSaved full report to: {out_path}")



=== Sepsis Setup Report (summary) ===
{
  "python": {
    "version": "3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:49:16) [MSC v.1929 64 bit (AMD64)]"
  },
  "packages": {
    "torch": {
      "ok": true,
      "version": "2.5.1"
    },
    "pandas": {
      "ok": true,
      "version": "2.3.1"
    },
    "numpy": {
      "ok": true,
      "version": "2.0.2"
    },
    "joblib": {
      "ok": true,
      "version": "1.5.1"
    },
    "sklearn": {
      "ok": true,
      "version": "1.6.1"
    },
    "stable_baselines3": {
      "ok": true,
      "version": "2.7.0"
    }
  },
  "paths": {
    "train_X": {
      "exists": true,
      "path": "C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\data\\processed\\sepsis\\train_X.csv"
    },
    "train_y": {
      "exists": true,
      "path": "C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\data\\processed\\sepsis\\train_y.csv"
    },
    "val_X": {
      "exists": 

C:\Users\vamsi\AppData\Local\Temp\ipykernel_7212\3678087947.py:125: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(expected_files["classifier"], map_location=

In [3]:
# 01_check_sb3_ppo_usage.py
# I wrote this to (1) search my code for SB3 PPO usage and (2) try to load the final agent via SB3.
# It prints a clear YES/NO and supporting evidence.

import os
import re
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
SRC = ROOT / "src"
TRAIN_FILES = [
    ROOT / "train_rl_agent_sepsis.py",
    SRC / "training" / "train_rl_agent_sepsis.py",
    SRC / "training" / "train_rl_agent.py",  # if I re-used a generic name
]

EVIDENCE = {
    "static_scan_hits": [],
    "ppo_logs_present": [],
    "sb3_import": None,
    "ppo_load": None
}

# 1) Static scan for 'stable_baselines3' and 'PPO('
pattern_import = re.compile(r"from\s+stable_baselines3\s+import\s+PPO|import\s+stable_baselines3", re.I)
pattern_ctor   = re.compile(r"\bPPO\s*\(", re.I)

def scan_file(path: Path):
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
        imp = bool(pattern_import.search(txt))
        ctor = bool(pattern_ctor.search(txt))
        if imp or ctor:
            EVIDENCE["static_scan_hits"].append({
                "file": str(path),
                "found_import": imp,
                "found_ctor": ctor
            })
    except Exception:
        pass

for cand in TRAIN_FILES:
    if cand.exists():
        scan_file(cand)

# 2) Check logs directory names for PPO_*
LOGS = ROOT / "logs"
if LOGS.exists():
    for p in LOGS.iterdir():
        if p.is_dir() and p.name.startswith("PPO_"):
            EVIDENCE["ppo_logs_present"].append(p.name)

# 3) Try to import SB3 and load the final agent
try:
    from stable_baselines3 import PPO
    EVIDENCE["sb3_import"] = {"ok": True}
    model_zip = ROOT / "models" / "rl_agent_sepsis_masked_final.zip"
    if model_zip.exists():
        try:
            _ = PPO.load(str(model_zip))
            EVIDENCE["ppo_load"] = {"ok": True, "path": str(model_zip)}
        except Exception as e:
            EVIDENCE["ppo_load"] = {"ok": False, "error": str(e), "path": str(model_zip)}
    else:
        EVIDENCE["ppo_load"] = {"ok": False, "error": "zip_missing", "path": str(model_zip)}
except Exception as e:
    EVIDENCE["sb3_import"] = {"ok": False, "error": str(e)}
    EVIDENCE["ppo_load"] = {"ok": False, "error": "sb3_not_available"}

# Verdict
is_sb3_ppo = (
    bool(EVIDENCE["static_scan_hits"]) or
    bool(EVIDENCE["ppo_logs_present"]) or
    (EVIDENCE["ppo_load"] and EVIDENCE["ppo_load"].get("ok"))
)

print("\n=== SB3/PPO Usage Check ===")
print("Likely SB3 PPO:", "YES" if is_sb3_ppo else "NO / INCONCLUSIVE")
print("\nEvidence:")
for k, v in EVIDENCE.items():
    print(f"- {k}: {v}")



=== SB3/PPO Usage Check ===
Likely SB3 PPO: YES

Evidence:
- static_scan_hits: []
- ppo_logs_present: ['PPO_1', 'PPO_10', 'PPO_11', 'PPO_12', 'PPO_13', 'PPO_14', 'PPO_15', 'PPO_16', 'PPO_17', 'PPO_18', 'PPO_2', 'PPO_3', 'PPO_4', 'PPO_5', 'PPO_6', 'PPO_7', 'PPO_8', 'PPO_9']
- sb3_import: {'ok': True}
- ppo_load: {'ok': False, 'error': "__init__() got an unexpected keyword argument 'use_sde'", 'path': 'C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\models\\rl_agent_sepsis_masked_final.zip'}


In [4]:
# 03_discover_features_and_costs.py
# I wrote this to (a) list my feature columns and count (from train_X.csv)
# and (b) search my repo for cost vectors and clinical constraint masks.

import os
import re
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path(os.getcwd()).resolve()
DATA_X = ROOT / "data" / "processed" / "sepsis" / "train_X.csv"

RESULT = {
    "features": {"ok": False},
    "costs": [],
    "constraints": [],
    "notes": []
}

# (a) Feature list and count
if pd is not None and DATA_X.exists():
    try:
        df = pd.read_csv(DATA_X, nrows=5)  # I only need columns
        cols = list(df.columns)
        RESULT["features"] = {"ok": True, "count": len(cols), "columns": cols[:60], "truncated": len(cols) > 60}
    except Exception as e:
        RESULT["features"] = {"ok": False, "error": str(e)}
else:
    RESULT["features"] = {"ok": False, "error": "pandas_missing_or_train_X_missing"}

# (b) Search for costs and masks in source files
SEARCH_DIRS = [
    ROOT,
    ROOT / "src",
    ROOT / "src" / "training",
    ROOT / "src" / "models"
]

# I’m looking for common keywords I used in this project.
cost_patterns = [
    r"test_costs\s*=",
    r"panel_costs\s*=",
    r"cost_vector\s*=",
    r"costs\s*=",
    r"cost\s*=",
]

mask_patterns = [
    r"mask\s*=",
    r"constraint[s]?\s*=",
    r"clinical_constraint[s]?\s*=",
    r"allowed_test[s]?\s*=",
]

def search_patterns(path: Path, patterns, tag):
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return
    for pat in patterns:
        for m in re.finditer(pat, txt):
            start = max(0, m.start() - 80)
            end = min(len(txt), m.end() + 200)
            snippet = txt[start:end].replace("\n", " ")
            yield {"file": str(path), "pattern": pat, "snippet": snippet[:300]}

for base in SEARCH_DIRS:
    if not base.exists():
        continue
    for path in base.rglob("*.py"):
        if path.name.startswith("."):
            continue
        for hit in search_patterns(path, cost_patterns, "cost"):
            RESULT["costs"].append(hit)
        for hit in search_patterns(path, mask_patterns, "mask"):
            RESULT["constraints"].append(hit)

# Hints
if not RESULT["costs"]:
    RESULT["notes"].append("No explicit cost vector found in .py files. It might be in a JSON/YAML, notebook, or generated at runtime.")
if not RESULT["constraints"]:
    RESULT["notes"].append("No explicit constraint mask found in .py files. It might be built from feature names or loaded from disk.")

# Print compact summary
print("\n=== Feature & Cost/Constraint Discovery ===")
print(f"- features.ok: {RESULT['features']['ok']}")
if RESULT["features"]["ok"]:
    print(f"- feature_count: {RESULT['features']['count']}")
    print(f"- first_columns: {RESULT['features']['columns']}")
else:
    print(f"- features_error: {RESULT['features'].get('error')}")

print(f"\n- cost_hits: {len(RESULT['costs'])}")
for h in RESULT["costs"][:5]:
    print(f"  * {h['file']} :: {h['pattern']} :: {h['snippet'][:160]}...")

print(f"\n- constraint_hits: {len(RESULT['constraints'])}")
for h in RESULT["constraints"][:5]:
    print(f"  * {h['file']} :: {h['pattern']} :: {h['snippet'][:160]}...")

if RESULT["notes"]:
    print("\nNotes:")
    for n in RESULT["notes"]:
        print(f"- {n}")



=== Feature & Cost/Constraint Discovery ===
- features.ok: True
- feature_count: 37
- first_columns: ['age', 'gender', 'heart_rate_mean', 'sbp_mean', 'dbp_mean', 'respiratory_rate_mean', 'spo2_mean', 'temperature_c_mean', 'heart_rate_min', 'sbp_min', 'dbp_min', 'respiratory_rate_min', 'spo2_min', 'temperature_c_min', 'heart_rate_max', 'sbp_max', 'dbp_max', 'respiratory_rate_max', 'spo2_max', 'temperature_c_max', 'abg_base_excess', 'cmp_lactate', 'abg_o2_saturation', 'abg_ph', 'cmp_aniongap', 'cmp_bicarbonate', 'cmp_creatinine', 'cmp_glucose', 'cmp_potassium', 'cmp_bun', 'cbc_hematocrit', 'cbc_hemoglobin', 'aptt_inr', 'cbc_platelet', 'aptt_ptt', 'cbc_rbc', 'cbc_wbc']

- cost_hits: 3
  * C:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\src\training\evaluate_agent_sepsis.py :: cost\s*= :: est_panels = sum(test_usage_counter.values()) / env.num_patients     avg_dollar_cost = sum(total_costs) / env.num_patients     accuracy = (num_correct_diagnoses...
  * C:\Use

In [6]:
# 03_discover_features_and_costs.py
# I wrote this to (a) list my feature columns and count (from train_X.csv)
# and (b) search my repo for cost vectors and clinical constraint masks.

import os
import re
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

ROOT = Path(os.getcwd()).resolve()
DATA_X = ROOT / "data" / "processed" / "sepsis" / "train_X.csv"

RESULT = {
    "features": {"ok": False},
    "costs": [],
    "constraints": [],
    "notes": []
}

# (a) Feature list and count
if pd is not None and DATA_X.exists():
    try:
        df = pd.read_csv(DATA_X, nrows=5)  # I only need columns
        cols = list(df.columns)
        RESULT["features"] = {"ok": True, "count": len(cols), "columns": cols[:60], "truncated": len(cols) > 60}
    except Exception as e:
        RESULT["features"] = {"ok": False, "error": str(e)}
else:
    RESULT["features"] = {"ok": False, "error": "pandas_missing_or_train_X_missing"}

# (b) Search for costs and masks in source files
SEARCH_DIRS = [
    ROOT,
    ROOT / "src",
    ROOT / "src" / "training",
    ROOT / "src" / "models"
]

# I’m looking for common keywords I used in this project.
cost_patterns = [
    r"test_costs\s*=",
    r"panel_costs\s*=",
    r"cost_vector\s*=",
    r"costs\s*=",
    r"cost\s*=",
]

mask_patterns = [
    r"mask\s*=",
    r"constraint[s]?\s*=",
    r"clinical_constraint[s]?\s*=",
    r"allowed_test[s]?\s*=",
]

def search_patterns(path: Path, patterns, tag):
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return
    for pat in patterns:
        for m in re.finditer(pat, txt):
            start = max(0, m.start() - 80)
            end = min(len(txt), m.end() + 200)
            snippet = txt[start:end].replace("\n", " ")
            yield {"file": str(path), "pattern": pat, "snippet": snippet[:300]}

for base in SEARCH_DIRS:
    if not base.exists():
        continue
    for path in base.rglob("*.py"):
        if path.name.startswith("."):
            continue
        for hit in search_patterns(path, cost_patterns, "cost"):
            RESULT["costs"].append(hit)
        for hit in search_patterns(path, mask_patterns, "mask"):
            RESULT["constraints"].append(hit)

# Hints
if not RESULT["costs"]:
    RESULT["notes"].append("No explicit cost vector found in .py files. It might be in a JSON/YAML, notebook, or generated at runtime.")
if not RESULT["constraints"]:
    RESULT["notes"].append("No explicit constraint mask found in .py files. It might be built from feature names or loaded from disk.")

# Print compact summary
print("\n=== Feature & Cost/Constraint Discovery ===")
print(f"- features.ok: {RESULT['features']['ok']}")
if RESULT["features"]["ok"]:
    print(f"- feature_count: {RESULT['features']['count']}")
    print(f"- first_columns: {RESULT['features']['columns']}")
else:
    print(f"- features_error: {RESULT['features'].get('error')}")

print(f"\n- cost_hits: {len(RESULT['costs'])}")
for h in RESULT["costs"][:5]:
    print(f"  * {h['file']} :: {h['pattern']} :: {h['snippet'][:160]}...")

print(f"\n- constraint_hits: {len(RESULT['constraints'])}")
for h in RESULT["constraints"][:5]:
    print(f"  * {h['file']} :: {h['pattern']} :: {h['snippet'][:160]}...")

if RESULT["notes"]:
    print("\nNotes:")
    for n in RESULT["notes"]:
        print(f"- {n}")



=== Feature & Cost/Constraint Discovery ===
- features.ok: True
- feature_count: 37
- first_columns: ['age', 'gender', 'heart_rate_mean', 'sbp_mean', 'dbp_mean', 'respiratory_rate_mean', 'spo2_mean', 'temperature_c_mean', 'heart_rate_min', 'sbp_min', 'dbp_min', 'respiratory_rate_min', 'spo2_min', 'temperature_c_min', 'heart_rate_max', 'sbp_max', 'dbp_max', 'respiratory_rate_max', 'spo2_max', 'temperature_c_max', 'abg_base_excess', 'cmp_lactate', 'abg_o2_saturation', 'abg_ph', 'cmp_aniongap', 'cmp_bicarbonate', 'cmp_creatinine', 'cmp_glucose', 'cmp_potassium', 'cmp_bun', 'cbc_hematocrit', 'cbc_hemoglobin', 'aptt_inr', 'cbc_platelet', 'aptt_ptt', 'cbc_rbc', 'cbc_wbc']

- cost_hits: 3
  * C:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\src\training\evaluate_agent_sepsis.py :: cost\s*= :: est_panels = sum(test_usage_counter.values()) / env.num_patients     avg_dollar_cost = sum(total_costs) / env.num_patients     accuracy = (num_correct_diagnoses...
  * C:\Use

In [7]:
# 04_inspect_rl_zip.py
# I wrote this to peek inside my saved SB3 zip (algo, version, hyperparams, policy_kwargs)
# WITHOUT instantiating PPO. This avoids the 'use_sde' crash and tells me exactly what's inside.

from pathlib import Path
from pprint import pprint

def main():
    root = Path.cwd()
    zip_path = root / "models" / "rl_agent_sepsis_masked_final.zip"
    if not zip_path.exists():
        print("zip_missing:", str(zip_path))
        return

    try:
        # I import lazily so the script still runs even if SB3 isn't present
        from stable_baselines3.common.save_util import load_from_zip_file
    except Exception as e:
        print("sb3_import_failed:", e)
        return

    # I only load metadata; I do NOT build the model here.
    data, params, _ = load_from_zip_file(str(zip_path), print_system_info=False)

    print("\n=== SB3 ZIP INSPECT ===")
    keys = sorted(list(data.keys()))
    print("data_keys:", keys)
    # hyperparameters dictionary is often stored under "hyperparameters" or "params"
    # In SB3 1.x it's 'hyperparameters' in data; in 2.x key names differ slightly. I list what I find.

    # Common fields I want to see:
    algo = data.get("algo", None)
    sb3_ver = data.get("sb3_version", None)
    policy_name = data.get("policy_name", None)
    hyper = data.get("hyperparameters", data.get("hyperparams", {}))
    policy_kwargs = hyper.get("policy_kwargs", {}) if isinstance(hyper, dict) else {}

    print("algo:", algo)
    print("sb3_version:", sb3_ver)
    print("policy_name:", policy_name)
    print("\n--- hyperparameters (top-level) ---")
    if isinstance(hyper, dict):
        # I print a curated subset so it’s readable
        interesting = {k: hyper.get(k) for k in [
            "n_steps","batch_size","gamma","learning_rate","clip_range",
            "gae_lambda","ent_coef","vf_coef","max_grad_norm","use_sde","sde_sample_freq"
        ] if k in hyper}
        pprint(interesting)
    else:
        print(type(hyper), "unexpected type")

    print("\n--- policy_kwargs ---")
    if isinstance(policy_kwargs, dict):
        for k, v in policy_kwargs.items():
            print(f"{k}: {v}")
    else:
        print(policy_kwargs)

    # I also show if 'use_sde' exists in saved hyperparams (the cause of 2.x load failure)
    if isinstance(hyper, dict) and "use_sde" in hyper:
        print("\nNOTE: 'use_sde' is present in saved hyperparameters -> model was trained with SB3 1.x.")

    print("\nDone.")

if __name__ == "__main__":
    main()



=== SB3 ZIP INSPECT ===
data_keys: ['_current_progress_remaining', '_episode_num', '_last_episode_starts', '_last_obs', '_last_original_obs', '_n_updates', '_num_timesteps_at_start', '_stats_window_size', '_total_timesteps', 'action_noise', 'action_space', 'batch_size', 'clip_range', 'clip_range_vf', 'ent_coef', 'ep_info_buffer', 'ep_success_buffer', 'gae_lambda', 'gamma', 'learning_rate', 'lr_schedule', 'max_grad_norm', 'n_envs', 'n_epochs', 'n_steps', 'normalize_advantage', 'num_timesteps', 'observation_space', 'policy_class', 'policy_kwargs', 'rollout_buffer_class', 'rollout_buffer_kwargs', 'sde_sample_freq', 'seed', 'start_time', 'target_kl', 'tensorboard_log', 'use_sde', 'verbose', 'vf_coef']
algo: None
sb3_version: None
policy_name: None

--- hyperparameters (top-level) ---
{}

--- policy_kwargs ---

Done.


In [8]:
# 04b_load_rl_zip_compat.py
# I wrote this to load my old PPO zip (trained with SB3 1.x) on SB3 2.x by stripping legacy kwargs.

from pathlib import Path
from tempfile import TemporaryDirectory

def main():
    from stable_baselines3.common.save_util import load_from_zip_file, save_to_zip_file
    from stable_baselines3 import PPO

    root = Path.cwd()
    src_zip = root / "models" / "rl_agent_sepsis_masked_final.zip"
    if not src_zip.exists():
        print("zip_missing:", str(src_zip))
        return

    data, params, _ = load_from_zip_file(str(src_zip), print_system_info=False)

    hyper = data.get("hyperparameters", data.get("hyperparams", {}))
    if not isinstance(hyper, dict):
        print("unexpected_hyper_type:", type(hyper))
        return

    # I remove keys that SB3 2.x PPO.__init__ does not accept.
    removed = []
    for legacy_key in ["use_sde", "sde_sample_freq", "n_episodes_rollout"]:
        if legacy_key in hyper:
            hyper.pop(legacy_key, None)
            removed.append(legacy_key)

    data["hyperparameters"] = hyper  # ensure I write back to the same key name

    print("Removed_legacy_keys:", removed)

    with TemporaryDirectory() as tmpd:
        tmp_zip = Path(tmpd) / "ppo_fixed.zip"
        # I repackage the archive in-memory dir
        save_to_zip_file(str(tmp_zip), data=data, params=params)
        # Now I try to actually construct a PPO model object
        try:
            model = PPO.load(str(tmp_zip), print_system_info=False)
            print("SUCCESS: Loaded PPO with patched hyperparameters.")
        except Exception as e:
            print("FAILED:", e)
            return

    print("All good. If needed, I can write a patched copy to disk; for now I only verified load.")

if __name__ == "__main__":
    main()


Removed_legacy_keys: []
FAILED: __init__() got an unexpected keyword argument 'use_sde'


In [9]:
"""
Final Evaluation Script for the Action-Masked RL Agent (v6.3 - Relocated)

v6.3: Paths have been updated to run correctly from the `notebooks/` directory.
"""
# --- Imports and Path Setup ---
import torch, numpy as np, pandas as pd, gymnasium as gym, os, sys
from collections import Counter
from sb3_contrib import MaskablePPO

# This path logic correctly adds the project's root directory to the system path,
# allowing us to import from the 'src' folder.
try:
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
except NameError:
    sys.path.append(os.path.abspath('..'))

# CORRECTED: The import path now explicitly looks inside the 'src.training' module
from src.training.train_rl_agent_sepsis import SepsisEnv, Config, TransformerPolicy

# --- Evaluation Function ---
def evaluate_sepsis_agent():
    print("--- Evaluating Final Masked Sepsis Agent ---")
    config = Config()
    PANEL_NAMES = {0: "CBC", 1: "CMP", 2: "ABG", 3: "aPTT"}
    
    # CORRECTED: Added '..' to the path to go up one level from `notebooks` to the root
    agent_path = os.path.join("..", config.MODEL_DIR, "rl_agent_sepsis_masked_final.zip")
    if not os.path.exists(agent_path):
        print(f"Error: Model not found at {agent_path}. Please train the agent first."); return

    # 1. Set up the Environment
    env = SepsisEnv(config)
    
    # CORRECTED: Added '..' to the paths for the data files
    data_path_prefix = os.path.join("..", config.PROCESSED_DATA_DIR)
    env.X_val = pd.read_csv(os.path.join(data_path_prefix, "test_X.csv"))
    env.y_val = pd.read_csv(os.path.join(data_path_prefix, "test_y.csv"))
    
    env.num_patients = len(env.X_val)
    print(f"Loaded TEST data with {env.num_patients} patients.")
    
    # 2. Load the Trained Agent
    model = MaskablePPO.load(agent_path)
    print("Trained agent loaded successfully.")

    # 3. Initialize Metrics
    total_rewards, total_costs, num_correct_diagnoses = [], [], 0
    test_usage_counter = Counter()

    # 4. Run Evaluation Loop
    for i in range(env.num_patients):
        obs, info = env.reset()
        done = False
        while not done:
            action_masks = env.action_masks()
            action, _states = model.predict(obs, action_masks=action_masks, deterministic=True)
            action = action.item()
            
            if action != env.DIAGNOSE_ACTION:
                total_costs.append(config.COST_MAPPING.get(action, 0))
                test_usage_counter[action] += 1
            
            obs, reward, done, truncated, info = env.step(action)
        
        if reward > 0: num_correct_diagnoses += 1
        total_rewards.append(reward)

    # 5. Calculate and Report Final Metrics
    avg_reward = np.mean(total_rewards)
    avg_test_panels = sum(test_usage_counter.values()) / env.num_patients
    avg_dollar_cost = sum(total_costs) / env.num_patients
    accuracy = (num_correct_diagnoses / env.num_patients) * 100

    print("\n" + "="*50 + "\n--- Final Performance Report ---\n" + "="*50)
    print(f"📈 Final Diagnostic Accuracy: {accuracy:.2f}%")
    print(f"💰 Average Financial Cost per Patient: ${avg_dollar_cost:.2f}")
    print(f"📉 Average Number of Tests Ordered: {avg_test_panels:.2f}")
    print(f"📊 Average Reward: {avg_reward:,.2f}")
    print("\n" + "="*50 + "\n--- Agent's Testing Strategy ---\n" + "="*50)
    for action_index, count in test_usage_counter.most_common():
        print(f"  - {PANEL_NAMES.get(action_index, 'Unknown')}: {count} times")
    print("="*50)

if __name__ == "__main__":
    evaluate_sepsis_agent()

--- Evaluating Final Masked Sepsis Agent ---
Loaded TEST data with 2517 patients.


C:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\src\training\train_rl_agent_sepsis.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.gai

Trained agent loaded successfully.

--- Final Performance Report ---
📈 Final Diagnostic Accuracy: 74.10%
💰 Average Financial Cost per Patient: $556.69
📉 Average Number of Tests Ordered: 2.28
📊 Average Reward: 4,819.23

--- Agent's Testing Strategy ---
  - aPTT: 2517 times
  - ABG: 2442 times
  - CBC: 505 times
  - CMP: 285 times


In [10]:
# 05_introspect_gain_signature.py
# I wrote this to print the exact __init__ signature of my GAIN Generator so I pass the right args.

import inspect
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

try:
    from src.models.gain import Generator
    sig = inspect.signature(Generator.__init__)
    print("\n=== GAIN.Generator __init__ signature ===")
    print(sig)
    print("\nParam details:")
    for name, param in sig.parameters.items():
        if name == "self":
            continue
        print(f"- {name}: kind={param.kind}, default={param.default}")
except Exception as e:
    print("gain_import_or_signature_failed:", e)

# Bonus: I also print recommended instantiation args using my 37-feature dimension.
try:
    import pandas as pd
    train_x = ROOT / "data" / "processed" / "sepsis" / "train_X.csv"
    n_features = 37
    if train_x.exists():
        df = pd.read_csv(train_x, nrows=1)
        n_features = df.shape[1]
    print(f"\nRecommended input_dim based on train_X.csv: {n_features}")
except Exception:
    print("\nCould not infer input_dim from CSV; defaulting to 37.")



=== GAIN.Generator __init__ signature ===
(self, input_dim)

Param details:
- input_dim: kind=POSITIONAL_OR_KEYWORD, default=<class 'inspect._empty'>

Recommended input_dim based on train_X.csv: 37


In [11]:
# 06_locate_costs_and_constraints.py
# I wrote this to locate cost vectors and clinical constraint masks in my env/training code.

from pathlib import Path
import re

ROOT = Path.cwd()
SEARCH_DIRS = [
    ROOT,
    ROOT / "src",
    ROOT / "src" / "training",
    ROOT / "src" / "models"
]

patterns = {
    "cost": [
        r"panel_costs\s*=",
        r"test_costs\s*=",
        r"cost_vector\s*=",
        r"dollar_cost[s]?\s*=",
        r"costs\s*=",
        r"cost\s*=",
    ],
    "mask": [
        r"clinical_constraint[s]?\s*=",
        r"allowed_test[s]?\s*=",
        r"constraint[s]?\s*=",
        r"mask\s*=",
        r"panel_mask\s*=",
        r"test_mask[s]?\s*="
    ],
    "env": [
        r"class\s+[A-Za-z0-9_]*Env\s*\(",
        r"gym\.Env",
        r"gymnasium\.Env",
        r"make\(",
    ],
    "moneyish": [
        r"\$|USD|dollar|rupee|cost_per|price|billing"
    ]
}

def scan(path: Path, pats):
    try:
        txt = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return []
    hits = []
    for tag, plist in pats.items():
        for pat in plist:
            for m in re.finditer(pat, txt, flags=re.I):
                start = max(0, m.start() - 120)
                end = min(len(txt), m.end() + 240)
                snippet = txt[start:end].replace("\n", " ")
                hits.append((tag, pat, snippet))
    return hits

results = []
for base in SEARCH_DIRS:
    if not base.exists():
        continue
    for p in base.rglob("*.py"):
        if p.name.startswith("."):
            continue
        hits = scan(p, patterns)
        if hits:
            results.append((str(p), hits))

print("\n=== Costs / Constraints / Env Locator ===")
for fname, hits in results[:40]:  # cap the print
    print(f"\nFile: {fname}")
    for tag, pat, snip in hits[:6]:
        print(f"  [{tag}] pattern='{pat}' -> ...{snip[:200]}...")

print("\nNote: If costs are built dynamically from feature panels, I’ll see them near an Env class or in policy/env constructors.")



=== Costs / Constraints / Env Locator ===

File: c:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\src\models\gain.py
  [mask] pattern='mask\s*=' -> ...m the Generator         h: the hint vector         """         input_cat = torch.cat([x, h], dim=1)         probability_mask = self.model(input_cat)         return probability_mask ...

File: c:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\src\training\evaluate_agent_sepsis.py
  [cost] pattern='dollar_cost[s]?\s*=' -> ...  avg_reward = np.mean(total_rewards)     avg_test_panels = sum(test_usage_counter.values()) / env.num_patients     avg_dollar_cost = sum(total_costs) / env.num_patients     accuracy = (num_correct_di...
  [cost] pattern='cost\s*=' -> ...eward = np.mean(total_rewards)     avg_test_panels = sum(test_usage_counter.values()) / env.num_patients     avg_dollar_cost = sum(total_costs) / env.num_patients     accuracy = (num_correct_diagnoses...
  [moneyish] pattern='\$|U

In [12]:
# 04c_patch_rl_zip_keys.py
# I wrote this to patch my old SB3 1.x PPO zip so it loads on SB3 2.x.
# It removes legacy top-level keys (like 'use_sde', 'sde_sample_freq') BEFORE model construction.
# It writes a new archive: models/rl_agent_sepsis_masked_final_patched.zip and verifies load.

from pathlib import Path
from tempfile import TemporaryDirectory

def main():
    try:
        from stable_baselines3.common.save_util import load_from_zip_file, save_to_zip_file
        from stable_baselines3 import PPO
    except Exception as e:
        print("stable_baselines3 import failed:", e)
        return

    root = Path.cwd()
    src_zip = root / "models" / "rl_agent_sepsis_masked_final.zip"
    if not src_zip.exists():
        print("zip_missing:", str(src_zip))
        return

    data, params, _ = load_from_zip_file(str(src_zip), print_system_info=False)

    # I remove legacy keys from TOP-LEVEL 'data' that PPO.__init__ no longer accepts in SB3 2.x.
    top_level_legacy = [
        "use_sde",
        "sde_sample_freq",
        # add any other observed unexpected kwargs here if needed
    ]
    removed = []
    for k in top_level_legacy:
        if k in data:
            removed.append(k)
            del data[k]

    # I also normalize where hyperparameters live to be consistent (optional but safe).
    # Some old zips use "hyperparameters", others "hyperparams".
    if "hyperparams" in data and "hyperparameters" not in data:
        data["hyperparameters"] = data["hyperparams"]

    # For safety, I DO NOT edit the hyperparameters dict here unless necessary.
    # The error proved the problem was at top-level kwargs.

    # Write a patched copy and verify I can load it with PPO.load.
    out_zip = root / "models" / "rl_agent_sepsis_masked_final_patched.zip"
    with TemporaryDirectory() as tmpd:
        tmp_zip = Path(tmpd) / "ppo_patched.zip"
        save_to_zip_file(str(tmp_zip), data=data, params=params)

        # Verify loading with SB3 2.x
        try:
            _ = PPO.load(str(tmp_zip), print_system_info=False)
            # If verification passes, I persist to disk.
            out_zip.write_bytes(tmp_zip.read_bytes())
            print("SUCCESS: Patched zip written to:", str(out_zip))
            print("Removed top-level legacy keys:", removed)
        except Exception as e:
            print("FAILED to load patched zip:", e)
            print("Removed top-level legacy keys:", removed)
            return

    print("Done.")

if __name__ == "__main__":
    main()


FAILED to load patched zip: __init__() got an unexpected keyword argument 'use_sde'
Removed top-level legacy keys: ['use_sde', 'sde_sample_freq']


In [13]:
# tools_load_gain_and_classifier.py
# I wrote this to load my GAIN Generator (input_dim only) and my classifier safely.
# I also run a tiny smoke-test forward for shapes (without gradients).

from pathlib import Path
import sys
import torch

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

def load_dims():
    # I infer input_dim from train_X.csv if available, else default to 37.
    import pandas as pd
    train_x = ROOT / "data" / "processed" / "sepsis" / "train_X.csv"
    if train_x.exists():
        df = pd.read_csv(train_x, nrows=1)
        return df.shape[1]
    return 37

def load_gain_generator():
    from src.models.gain import Generator
    gen_path = ROOT / "models" / "generator_sepsis.pth"
    input_dim = load_dims()
    gen = Generator(input_dim=input_dim)
    sd = torch.load(gen_path, map_location="cpu")
    # My saved file is weights-only; I allow strict=False in case of extra keys.
    gen.load_state_dict(sd, strict=False)
    gen.eval()
    return gen, input_dim

def load_classifier():
    from src.models.classifier import PreliminaryClassifier
    clf_path = ROOT / "models" / "classifier_sepsis.pth"
    input_dim = load_dims()
    clf = PreliminaryClassifier(input_dim=input_dim)
    sd = torch.load(clf_path, map_location="cpu")
    clf.load_state_dict(sd, strict=False)
    clf.eval()
    return clf, input_dim

if __name__ == "__main__":
    try:
        gen, d = load_gain_generator()
        print(f"GAIN OK: loaded with input_dim={d}")
        # smoke test (I build a fake mask of ones, because my forward usually expects (x, mask) or similar)
        with torch.no_grad():
            x = torch.randn(2, d)
            try:
                # Try common GAIN call patterns
                mask = torch.ones_like(x)
                _ = gen(x, mask)  # if my __call__ expects (x, mask)
                print("GAIN smoke-test forward: (x, mask) call succeeded.")
            except TypeError:
                # fallback: try single-arg forward
                _ = gen(x)
                print("GAIN smoke-test forward: (x) call succeeded.")
    except Exception as e:
        print("GAIN load failed:", e)

    try:
        clf, d = load_classifier()
        print(f"Classifier OK: loaded with input_dim={d}")
        with torch.no_grad():
            x = torch.randn(2, d)
            _ = clf(x)
            print("Classifier smoke-test forward: OK.")
    except Exception as e:
        print("Classifier load failed:", e)


GAIN OK: loaded with input_dim=37
GAIN smoke-test forward: (x, mask) call succeeded.
Classifier OK: loaded with input_dim=37
Classifier smoke-test forward: OK.


C:\Users\vamsi\AppData\Local\Temp\ipykernel_7212\734413277.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(gen_path, map_location="cpu")
C:\Users\vamsi\

In [14]:
# 02_probe_transformer_integration.py
# I wrote this to discover whether my Transformer is used as a custom policy or as a BaseFeaturesExtractor.
# It searches code and tries to import the transformer module to list classes.

import os
import re
import inspect
from pathlib import Path
import importlib.util
import sys

ROOT = Path(os.getcwd()).resolve()
sys.path.insert(0, str(ROOT))

SRC_MODELS = ROOT / "src" / "models"
TRAIN_DIR  = ROOT / "src" / "training"
CANDIDATES = [
    ROOT / "train_rl_agent_sepsis.py",
    TRAIN_DIR / "train_rl_agent_sepsis.py",
    TRAIN_DIR / "train_rl_agent.py"
]

FINDINGS = {"transformer_module": {}, "policy_usage": [], "feature_extractor_usage": [], "policy_args": []}

# 1) Try to import transformer module and introspect
trans_path = SRC_MODELS / "transformer.py"
if trans_path.exists():
    try:
        spec = importlib.util.spec_from_file_location("src.models.transformer", str(trans_path))
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)  # type: ignore
        classes = []
        for name, obj in inspect.getmembers(mod, inspect.isclass):
            classes.append(name)
        FINDINGS["transformer_module"] = {"ok": True, "path": str(trans_path), "classes": classes}
    except Exception as e:
        FINDINGS["transformer_module"] = {"ok": False, "error": str(e), "path": str(trans_path)}
else:
    FINDINGS["transformer_module"] = {"ok": False, "error": "transformer.py not found", "path": str(trans_path)}

# 2) Static scan training scripts for policy/features-extractor hooks
re_policy_kw = re.compile(r"policy\s*=\s*['\"]([A-Za-z0-9_]+)['\"]|policy\s*=\s*([A-Za-z0-9_]+)", re.I)
re_features_extractor = re.compile(r"BaseFeaturesExtractor|features_extractor|features_extractor_class", re.I)
re_custom_policy_class = re.compile(r"class\s+([A-Za-z0-9_]+Policy)\s*\(", re.I)
re_policy_kwargs = re.compile(r"policy_kwargs\s*=\s*{", re.I)

for fn in CANDIDATES:
    if not fn.exists():
        continue
    txt = fn.read_text(encoding="utf-8", errors="ignore")

    # Find explicit policy argument usage
    for m in re_policy_kw.finditer(txt):
        FINDINGS["policy_usage"].append({"file": str(fn), "match": m.group(0)})

    # Find features-extractor mentions
    if re_features_extractor.search(txt):
        FINDINGS["feature_extractor_usage"].append({"file": str(fn), "hint": "mentions features extractor"})

    # Look for custom policy class definitions
    for m in re_custom_policy_class.finditer(txt):
        FINDINGS["policy_usage"].append({"file": str(fn), "custom_policy_class": m.group(1)})

    # Capture policy_kwargs presence (often where features_extractor_class is passed)
    if re_policy_kwargs.search(txt):
        FINDINGS["policy_args"].append({"file": str(fn), "has_policy_kwargs_dict": True})

print("\n=== Transformer Integration Probe ===")
print("Transformer module:", FINDINGS["transformer_module"])
print("Policy usage hits:", FINDINGS["policy_usage"])
print("Features extractor hints:", FINDINGS["feature_extractor_usage"])
print("Policy kwargs detected:", FINDINGS["policy_args"])
print("\nNotes:")
print("- If 'features_extractor' or 'features_extractor_class' appears in policy_kwargs, my Transformer is likely a features extractor.")
print("- If a custom *Policy class is defined/used, the Transformer might be baked into a custom policy.")



=== Transformer Integration Probe ===
Transformer module: {'ok': True, 'path': 'C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\src\\models\\transformer.py', 'classes': ['TransformerSelector']}
Policy usage hits: [{'file': 'C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\src\\training\\train_rl_agent_sepsis.py', 'custom_policy_class': 'TransformerPolicy'}]
Features extractor hints: [{'file': 'C:\\Users\\vamsi\\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\\src\\training\\train_rl_agent_sepsis.py', 'hint': 'mentions features extractor'}]
Policy kwargs detected: []

Notes:
- If 'features_extractor' or 'features_extractor_class' appears in policy_kwargs, my Transformer is likely a features extractor.
- If a custom *Policy class is defined/used, the Transformer might be baked into a custom policy.


In [15]:
# 04d_patch_rl_zip_deep.py
# I wrote this to *deeply* patch my SB3 1.x PPO zip so it loads on SB3 2.x.
# It removes legacy kwargs ('use_sde', 'sde_sample_freq') from any place SB3 1.x might store them:
# - top-level data dict
# - rollout_buffer_kwargs
# - policy_kwargs (rare)
# - any nested dict under known keys
# Then it writes models/rl_agent_sepsis_masked_final_patched_deep.zip and verifies load.

from pathlib import Path
from tempfile import TemporaryDirectory

LEGACY_KEYS = {"use_sde", "sde_sample_freq", "n_episodes_rollout"}  # expand if loader complains further
DICT_KEYS_TO_SCAN = [
    "policy_kwargs",
    "rollout_buffer_kwargs",
    "hyperparameters",
    "hyperparams",
    "lr_schedule",      # sometimes dict-like in old saves
]

def strip_legacy(d):
    if not isinstance(d, dict):
        return
    for k in list(d.keys()):
        if k in LEGACY_KEYS:
            d.pop(k, None)
    # Recurse shallowly into known nested dicts
    for nk in DICT_KEYS_TO_SCAN:
        if nk in d and isinstance(d[nk], dict):
            strip_legacy(d[nk])

def main():
    try:
        from stable_baselines3.common.save_util import load_from_zip_file, save_to_zip_file
        from stable_baselines3 import PPO
    except Exception as e:
        print("stable_baselines3 import failed:", e); return

    root = Path.cwd()
    src_zip = root / "models" / "rl_agent_sepsis_masked_final.zip"
    if not src_zip.exists():
        print("zip_missing:", str(src_zip)); return

    data, params, _ = load_from_zip_file(str(src_zip), print_system_info=False)

    # 1) Top-level strip
    strip_legacy(data)

    # 2) Some SB3 1.x zips keep all ctor kwargs directly at top-level of `data`
    #    (things like 'n_steps', 'batch_size', 'use_sde' etc). I explicitly drop known legacy keys there too.
    for k in list(data.keys()):
        if k in LEGACY_KEYS:
            data.pop(k, None)

    # 3) Normalize hyperparameter key name
    if "hyperparams" in data and "hyperparameters" not in data:
        data["hyperparameters"] = data["hyperparams"]

    out_zip = root / "models" / "rl_agent_sepsis_masked_final_patched_deep.zip"
    with TemporaryDirectory() as tmpd:
        tmp_zip = Path(tmpd) / "ppo_patched_deep.zip"
        save_to_zip_file(str(tmp_zip), data=data, params=params)

        try:
            # Verify SB3 2.x can load the patched zip
            _ = PPO.load(str(tmp_zip), print_system_info=False)
            out_zip.write_bytes(tmp_zip.read_bytes())
            print("SUCCESS: Patched zip written to:", str(out_zip))
        except Exception as e:
            print("FAILED to load patched zip:", e)
            # If it still fails on a different legacy key, tell me the key so I can add it to LEGACY_KEYS.
            legacy_in_data = [k for k in data.keys() if k in LEGACY_KEYS]
            print("Residual legacy keys at top-level:", legacy_in_data)
            return

    print("Done.")

if __name__ == "__main__":
    main()


FAILED to load patched zip: __init__() got an unexpected keyword argument 'use_sde'
Residual legacy keys at top-level: []


In [16]:
# 07_dump_sepsis_env_metadata.py
# I wrote this to introspect my SepsisEnv for panel groups, masks, and costs.
# It avoids training/eval; it only imports and inspects attributes and (if possible) a lightweight instance.

import sys
from pathlib import Path
import inspect

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

def main():
    mod = None
    try:
        mod = __import__("src.training.train_rl_agent_sepsis", fromlist=["*"])
    except Exception as e:
        print("import_failed:", e); return

    # 1) Locate SepsisEnv class
    SepsisEnv = getattr(mod, "SepsisEnv", None)
    if SepsisEnv is None or not inspect.isclass(SepsisEnv):
        print("SepsisEnv_not_found_in_module"); return

    print("=== SepsisEnv found ===")

    # 2) List interesting module-level attributes likely related to costs/panels
    interesting = {}
    for name, val in mod.__dict__.items():
        if any(key in name.upper() for key in ["COST", "COSTS", "PRICE", "TEST", "PANEL", "GROUP", "MASK"]):
            # Avoid dumping huge arrays
            try:
                s = str(val)
                s = s if len(s) < 300 else s[:300] + "...(truncated)"
                interesting[name] = s
            except Exception:
                interesting[name] = f"<unprintable type {type(val)}>"

    print("\n--- Module-level candidates (COST/TEST/PANEL/GROUP/MASK) ---")
    for k, v in sorted(interesting.items()):
        print(f"{k} = {v}")

    # 3) Try to inspect SepsisEnv.__init__ signature (to see required config keys)
    sig = inspect.signature(SepsisEnv.__init__)
    print("\nSepsisEnv.__init__ signature:", sig)

    # 4) Attempt a *minimal* instantiation if there is an obvious CONFIG in module
    #    I try to find a dict named CONFIG or something similar to pass in.
    candidates = [name for name in dir(mod) if name.upper() in ("CONFIG", "SEPSIS_CONFIG", "DEFAULT_CONFIG")]
    env = None
    for cname in candidates:
        cfg = getattr(mod, cname, None)
        if isinstance(cfg, dict):
            try:
                env = SepsisEnv(cfg)
                print(f"\n[OK] Instantiated env with {cname}.")
                break
            except Exception as e:
                print(f"\n[WARN] Could not instantiate with {cname}: {e}")

    # 5) If I have an env, dump key runtime attributes
    if env is not None:
        attrs = {}
        for name in dir(env):
            if any(tag in name.lower() for tag in ["cost", "panel", "group", "mask", "action", "num_", "test"]):
                try:
                    val = getattr(env, name)
                    s = str(val)
                    s = s if len(s) < 300 else s[:300] + "...(truncated)"
                    attrs[name] = s
                except Exception:
                    pass

        print("\n--- Env runtime attributes (filtered) ---")
        for k, v in sorted(attrs.items()):
            print(f"{k}: {v}")

    print("\nDone.")

if __name__ == "__main__":
    main()


=== SepsisEnv found ===

--- Module-level candidates (COST/TEST/PANEL/GROUP/MASK) ---
MaskablePPO = <class 'sb3_contrib.ppo_mask.ppo_mask.MaskablePPO'>

SepsisEnv.__init__ signature: (self, config)

Done.


In [17]:
# 04e_load_maskableppo_compat.py
# I wrote this to load my old MaskablePPO zip (trained with SB3 1.x + sb3-contrib) in SB3 2.x.
# It strips legacy kwargs (like 'use_sde', 'sde_sample_freq') from any saved metadata path,
# then loads with sb3_contrib.ppo_mask.MaskablePPO.

from pathlib import Path
from tempfile import TemporaryDirectory

LEGACY_KEYS = {"use_sde", "sde_sample_freq", "n_episodes_rollout"}

DICT_KEYS_TO_SCAN = [
    "policy_kwargs",
    "rollout_buffer_kwargs",
    "hyperparameters",
    "hyperparams",
    "lr_schedule",
]

def strip_legacy(d):
    if not isinstance(d, dict):
        return
    for k in list(d.keys()):
        if k in LEGACY_KEYS:
            d.pop(k, None)
    for nk in DICT_KEYS_TO_SCAN:
        if nk in d and isinstance(d[nk], dict):
            strip_legacy(d[nk])

def main():
    try:
        from stable_baselines3.common.save_util import load_from_zip_file, save_to_zip_file
        from sb3_contrib.ppo_mask import MaskablePPO
    except Exception as e:
        print("imports_failed:", e); return

    root = Path.cwd()
    src_zip = root / "models" / "rl_agent_sepsis_masked_final.zip"
    if not src_zip.exists():
        print("zip_missing:", str(src_zip)); return

    data, params, _ = load_from_zip_file(str(src_zip), print_system_info=False)

    # Deep strip
    strip_legacy(data)
    for k in list(data.keys()):
        if k in LEGACY_KEYS:
            data.pop(k, None)

    if "hyperparams" in data and "hyperparameters" not in data:
        data["hyperparameters"] = data["hyperparams"]

    out_zip = root / "models" / "rl_agent_sepsis_masked_final_maskable_patched.zip"
    with TemporaryDirectory() as tmpd:
        tmp_zip = Path(tmpd) / "maskable_patched.zip"
        save_to_zip_file(str(tmp_zip), data=data, params=params)
        try:
            _ = MaskablePPO.load(str(tmp_zip), print_system_info=False)
            out_zip.write_bytes(tmp_zip.read_bytes())
            print("SUCCESS: Patched zip written to:", str(out_zip))
        except Exception as e:
            print("FAILED:", e)
            return
    print("Done.")

if __name__ == "__main__":
    main()


SUCCESS: Patched zip written to: c:\Users\vamsi\cost-effective-diagnosis-using-a-transformer-driven-rl-policy\models\rl_agent_sepsis_masked_final_maskable_patched.zip
Done.
